In [ ]:
import io
import json
import time
import requests
from google.colab import files

SUBMIT_URL = "https://trangden.vn/agentsee/api/bai-tap/3494/a090b38ae7f995af2caaa2a74c887e89/tuan-1/bai-7/cau-3"
VERIFY_URL = SUBMIT_URL + "/verify"
MAP_URL = "https://leafletjs.com"
AGENT_NAME = "Codex"

def fetch_runtime_info():
    services = [
        "https://ipinfo.io/json",
        "https://ipapi.co/json/",
        "https://httpbin.org/ip",
    ]
    for url in services:
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            return {"source": url, "data": response.json()}
        except Exception:
            continue
    return {"source": None, "data": {"error": "Khong lay duoc thong tin IP runtime"}}

def print_json(title, payload):
    print(f"\n=== {title} ===")
    print(json.dumps(payload, ensure_ascii=False, indent=2))

runtime_info = fetch_runtime_info()
print_json("IP / Geo hien tai", runtime_info)

uploaded = files.upload()
if not uploaded:
    raise SystemExit("Chua upload file anh.")

file_name, content = next(iter(uploaded.items()))
print(f"\nDa nhan file: {file_name} ({len(content):,} bytes)")
print(f"Map URL: {MAP_URL}")

files_payload = {
    "screenshot": (file_name, io.BytesIO(content), "image/png"),
}
data_payload = {
    "map_url": MAP_URL,
    "agent": AGENT_NAME,
}

submit_response = requests.post(SUBMIT_URL, files=files_payload, data=data_payload, timeout=60)
submit_response.raise_for_status()
print_json("Ket qua submit", submit_response.json())

for attempt in range(1, 4):
    time.sleep(5)
    verify_response = requests.post(VERIFY_URL, timeout=60)
    verify_response.raise_for_status()
    verify_result = verify_response.json()
    print_json(f"Ket qua verify lan {attempt}", verify_result)
    if verify_result.get("valid") is True:
        print("\nPASS roi. Xong viec.")
        break
else:
    print("\nVan chua pass sau 3 lan verify.")
    print("Runtime Colab nay co the khong nam o My, hoac de dang cham metadata khac.")
